In [ ]:
import torch
from ultralytics import YOLO
import os
from kaggle.api.kaggle_api_extended import KaggleApi
from pathlib import Path
import yaml
import glob
import cv2
from roboflow import Roboflow
import kagglehub

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

  Using cached kagglehub-1.0.0-py3-none-any.whl.metadata (40 kB)
Using cached kagglehub-1.0.0-py3-none-any.whl (70 kB)
Note: you may need to restart the kernel to use updated packages.
True
NVIDIA GeForce RTX 4060 Laptop GPU


IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html


## Descarga del dataset de entrenamiento y validación

In [2]:
ruta_dataset = "Gun-1"

rf = Roboflow(api_key="SXZzIRF9BDmUj3Iyp8NJ")
project = rf.workspace("pedros-workspace-lpjzj").project("gun-soqmi-k9jw6")
version = project.version(1)

if not os.path.exists(ruta_dataset):
    dataset = version.download("yolo26", location=ruta_dataset)
    ubicacion = dataset.location
else:
    ubicacion = os.path.abspath(ruta_dataset)

print("Dataset descargado en:", ubicacion)

loading Roboflow workspace...
loading Roboflow project...
Dataset descargado en: c:\Users\bleik\Documents\GitHub\IABD\PIA\VisionArtificial\proyecto\Gun-1


## Entrenamiento del modelo

In [3]:
# model = YOLO('yolo26x.pt') # Modelo grande
model = YOLO('yolo26m.pt') # Modelo intermedio
# model = YOLO('yolo26n.pt') # Modelo pequeño

results = model.train(
    data='Gun-1/data.yaml',
    epochs=30,
    imgsz=640,
    device=0,
    batch=8,
    name='guns'
)

New https://pypi.org/project/ultralytics/8.4.21 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.14  Python-3.13.12 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=Gun-1/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=guns3, nb

## Métricas de rendimiento

In [ ]:
modelo_entrenado = YOLO('./best.pt')

metricas = modelo_entrenado.val()

print(f"Precisión: {metricas.box.p.mean()}")
print(f"Recall: {metricas.box.r.mean()}")
print(f"mAP@50: {metricas.box.map50.mean()}")
print(f"mAP@50-95: {metricas.box.map.mean()}")

Ultralytics 8.4.14  Python-3.13.12 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
YOLO26m summary (fused): 132 layers, 20,350,223 parameters, 0 gradients, 67.8 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 669.3773.2 MB/s, size: 113.7 KB)
val: Scanning C:\Users\bleik\Documents\GitHub\IABD\PIA\VisionArtificial\proyecto\Gun-1\valid\labels.cache... 4041 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 4041/4041 1.5Git/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 253/253 2.8it/s 1:31<0.4ss
                   all       4041       4399      0.951      0.916      0.967      0.649
Speed: 0.5ms preprocess, 19.9ms inference, 0.0ms loss, 0.1ms postprocess per image
Results saved to C:\Users\bleik\Documents\GitHub\IABD\runs\detect\val4
Precisión: 0.9513521085259568
Recall: 0.9157786792429055
mAP@50: 0.9666529534384527
mAP@50-95: 0.649064837112798


## Prueba con video

In [ ]:
ruta_video = 'gunScenenes.mp4'
captura = cv2.VideoCapture(ruta_video)

ancho = int(captura.get(cv2.CAP_PROP_FRAME_WIDTH))
alto = int(captura.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(captura.get(cv2.CAP_PROP_FPS))

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
salida = cv2.VideoWriter('resultado_armas.mp4', fourcc, fps, (ancho, alto))

while captura.isOpened():
    exito, fotograma = captura.read()
    
    if not exito:
        break

    resultados = modelo_entrenado(fotograma)
    
    fotograma_anotado = resultados[0].plot()
    
    salida.write(fotograma_anotado)

captura.release()
salida.release()
print("Procesamiento de video completado. Archivo guardado como 'resultado_armas.mp4'.")


0: 384x640 (no detections), 70.8ms
Speed: 3.2ms preprocess, 70.8ms inference, 0.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 12.8ms
Speed: 1.6ms preprocess, 12.8ms inference, 0.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 13.3ms
Speed: 1.8ms preprocess, 13.3ms inference, 0.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 gun, 13.6ms
Speed: 1.9ms preprocess, 13.6ms inference, 1.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 gun, 12.8ms
Speed: 1.8ms preprocess, 12.8ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 gun, 13.4ms
Speed: 1.6ms preprocess, 13.4ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 gun, 16.6ms
Speed: 2.0ms preprocess, 16.6ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 14.6ms
Speed: 1.6ms preprocess, 14.6ms inference, 0.3ms postprocess per image at

## Prueba con otro dataset

In [7]:
path = kagglehub.dataset_download("ugorjiir/gun-detection")

print("Ruta base del dataset:", path)

directorio_imagenes = path

for raiz, directorios, archivos in os.walk(path):
    if any(archivo.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.webp')) for archivo in archivos):
        directorio_imagenes = raiz
        break

print("Directorio de imágenes encontrado:", directorio_imagenes)

resultados = modelo_entrenado.predict(
    source=directorio_imagenes,
    save=True,
    project='evaluacion_kaggle',
    name='predicciones_gun_detection'
)

Ruta base del dataset: C:\Users\bleik\.cache\kagglehub\datasets\ugorjiir\gun-detection\versions\1
Directorio de imágenes encontrado: C:\Users\bleik\.cache\kagglehub\datasets\ugorjiir\gun-detection\versions\1\Gunmen Dataset\All

WARNING 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

image 1/1310 C:\Users\bleik\.cache\kagglehub\datasets\ugorjiir\gun-detection\versions\1\Gunmen Dataset\All\02fb4ea38c03afae.jpg: 448x640 1 gun, 63.2ms
image 2/1310 C:\Users\bleik\.cache\kagglehub\datasets\ugorjiir\gun-detection\versio